# Datamine PICTED Process Example

This notebook demonstrates how to configure and run the **`picted`** process wrapper in `dmstudio`.

## Process Description

## PICTED

Interactive display and manipulation of Datamine plot files to produce composite plots.

PICTED allows different plot files to be displayed concurrently, and permits rescaling and positioning of individual plots. The composite picture created in this way may be saved to an output *.dm format plot file.

### Process Options

On running the process, the following options are displayed:

## 1. SET PLOT DIMENSIONS IN MM

## 2. DISPLAY A FILE (AND ADD TO ACTIVE LIST)

## 3. SET/CHANGE OFFSETS/SCALES

## 4. REMOVE FILE FROM ACTIVE LIST

## 5. SAVE DISPLAY INTO NEW PLOT FILE

## 6. REFRESH DISPLAY

## 7. HARDCOPY CURRENT DISPLAY

## 8. EXIT

Each of the above options generates subsidiary interactions:

1. For SET PLOT DIMENSIONS IN MM:

## CURRENT PLOT DIMENSIONS ARE : X = XXX.XX, Y = YYY.YY

## >>> SET NOMINAL PLOT SIZE IN MM>

X > Supply X plot dimension.

Y > Supply Y plot dimension.

FILE > Supply plot file name.

## >>> EXPAND (>1) OR REDUCE (<1) FACTOR >

X > Supply X scaling factor.

Y > Supply Y scaling factor.

## >>> X AND Y OFFSETS IN MM --

X > Supply X offset from picture origin for the local origin of

the current plot file.

Y > Supply Y offset from picture origin for the local origin of

the current plot file.

## SET OR CHANGE SCALES AND OFFSETS

## >>> N FILES ACTIVE ---

>>>================================================

## >>> FILENAME FILE LIMITS SCALED BY OFFSETS

## >>> X Y X Y X Y

>>>================================================

>>> xxxxxxxx nn.n nn.n nn.n nn.n nn.n nn.n

2. For DISPLAY A FILE (AND ADD TO ACTIVE LIST):

The Project Browser is displayed in order to select the required plot file.

3. For SET/CHANGE OFFSETS/SCALES:

Scales and offsets are prompted for, as for option 2.

4. For REMOVE FILE FROM ACTIVE LIST:

The Project Browser dialog is displayed in order to select the required plot file.

## REMOVE FILE FROM ACTIVE LIST

## >>> CURRENTLY ACTIVE FILES ARE --

>>> xxxxxxxx

>>> yyyyyyyy

5. For SAVE DISPLAY INTO NEW PLOT FILE:

The Project Browser is displayed in order to define the required output plot file location and name.

6. For REFRESH DISPLAY:

The Graphics widow display is cleared and regenerated from the plot files in the active list.

7. For HARDCOPY CURRENT DISPLAY:

This function is not available.

8. For EXIT

The **PICTED** process is terminated.

### Input Files:

### Output Files:

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('picted')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute picted
print("Running picted...")
dm_fil.picted(
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("picted execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_picted_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")